In [26]:
import os
import pandas as pd
from sqlalchemy import create_engine

# 1. Configuración de Credenciales
user = "jcardonr"
password = "6nxx2yVWaKXyiF1cZbfS*"
host = "10.250.5.36"
db_name = "system_savia"
port = "3306"

# Crear el motor de conexión
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}")

# 2. Definición de la Query (Limpia)
query_auditoria = """
SELECT 
    f.id as factura_id,
    f.valor_factura,
    f.valor_inicial_glosa,
    f.fecha_radicacion,
    f.fecha_prestacion,
    f.cnt_prestadores_id,
    f.mae_tipo_contrato_valor,
    f.pbs,
    p.razon_social  AS prestador_nombre,
    p.numero_documento  AS prestador_nit,
    a.fecha_nacimiento,
    a.genero  as genero
FROM cm_facturas f
LEFT JOIN cnt_prestadores p ON f.cnt_prestadores_id = p.id
-- Buscamos el afiliado en los detalles para mayor precisión
LEFT JOIN (
    SELECT cm_facturas_id, MAX(aseg_afiliados_id) as afiliado_id 
    FROM cm_detalles 
    GROUP BY cm_facturas_id
) d ON f.id = d.cm_facturas_id
LEFT JOIN aseg_afiliados a ON d.afiliado_id = a.id
WHERE f.fecha_radicacion >= DATE_SUB(NOW(), INTERVAL 3 MONTH)
LIMIT 50000;
"""

# 3. Proceso de extracción y guardado
try:
    print(f"Conectando al servidor {host}...")
    
    # Cargar datos a memoria (DataFrame)
    df_raw = pd.read_sql(query_auditoria, con=engine)
    
    # Definir ruta y asegurar que la carpeta existe
    file_path = r'D:\Users\jcardonr\Documents\eafit\Codigo\data\extracto_auditoria_3meses.csv'
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    # Exportar a CSV
    df_raw.to_csv(file_path, index=False, encoding='utf-8-sig')
    
    print("-" * 30)
    print(f"¡Éxito! Registros descargados: {len(df_raw)}")
    print(f"Ubicación: {file_path}")
    print("-" * 30)

except Exception as e:
    print(f"Ocurrió un error: {e}")

Conectando al servidor 10.250.5.36...
------------------------------
¡Éxito! Registros descargados: 50000
Ubicación: D:\Users\jcardonr\Documents\eafit\Codigo\data\extracto_auditoria_3meses.csv
------------------------------


In [24]:
# Consulta de conteo total para validar el universo de datos
count_query = "SELECT COUNT(*) as total FROM _tmp_facturas_ia WHERE fecha_radicacion >= '2025-01-01'"
total_en_db = pd.read_sql(count_query, con=engine)
print(f"Total real de registros en la tabla para el último año: {total_en_db['total'][0]}")

Total real de registros en la tabla para el último año: 2000


In [23]:
import pandas as pd
import numpy as np

# 1. Cargar el extracto local
df = pd.read_csv(r'D:\Users\jcardonr\Documents\eafit\Codigo\data\extracto_auditoria_3meses.csv')

# 2. Definir el Target (Variable a predecir)
# 1 si hubo glosa inicial, 0 si no
df['target_glosa'] = (df['valor_inicial_glosa'] > 0).astype(int)

# 3. Creación de Features (Ingeniería de variables)

# A. Tiempo de Radicación (Días entre prestación y radicación)
df['fecha_prestacion'] = pd.to_datetime(df['fecha_prestacion'])
df['fecha_radicacion'] = pd.to_datetime(df['fecha_radicacion'])
df['dias_espera_radicacion'] = (df['fecha_radicacion'] - df['fecha_prestacion']).dt.days

# B. Ratio de Copago
# ¿Qué tanto del total es copago? (Evitando división por cero)
df['ratio_copago'] = df['valor_copago'] / (df['valor_factura'] + 1)

# C. Variable de Tipo de Contrato (One-Hot Encoding manual)
# Transformamos "EVENTO", "CAPITA", etc., en números para el modelo
df['is_evento'] = (df['mae_tipo_contrato_valor'] == 'EVENTO').astype(int)

# 4. Limpieza de datos específicos de MySQL (Bits/Booleans)
# Colab a veces lee los bits como b'\x00'. Los pasamos a 0 y 1.
def clean_bit(x):
    if isinstance(x, str) and 'x01' in x: return 1
    if isinstance(x, str) and 'x00' in x: return 0
    return x

df['multiusuario'] = df['multiusuario'].apply(clean_bit)
df['pbs'] = df['pbs'].apply(clean_bit)

# 5. Selección de columnas para el entrenamiento inicial
features_interes = [
    'valor_factura', 'valor_bruto', 'valor_copago', 'dias_espera_radicacion',
    'is_evento', 'multiusuario', 'pbs', 'target_glosa'
]

df_final = df[features_interes].fillna(0)

print("### Dataset listo para entrenamiento ###")
print(f"Total registros: {len(df_final)}")
print(f"Casos con Glosa detectados: {df_final['target_glosa'].sum()}")
print(df_final.head())

# Guardar el dataset procesado
df_final.to_csv(r'D:\Users\jcardonr\Documents\eafit\Codigo\data\dataset_preprocesado.csv', index=False)

### Dataset listo para entrenamiento ###
Total registros: 2000
Casos con Glosa detectados: 0
   valor_factura  valor_bruto  valor_copago  dias_espera_radicacion  \
0       158960.0     158960.0           0.0                       2   
1        28640.0      28640.0           0.0                       2   
2        58960.0      58960.0           0.0                       2   
3        76500.0      76500.0           0.0                       2   
4        78600.0      78600.0           0.0                       2   

   is_evento  multiusuario  pbs  target_glosa  
0          1             0    1             0  
1          1             0    1             0  
2          1             0    1             0  
3          1             0    1             0  
4          1             0    1             0  


In [25]:
# Lista de tablas candidatas a ser el histórico
tablas_a_validar = ['cm_fe_rips_cargas', 'cm_rips_cargas', 'aseg_afiliados']

print("--- Verificando volumen de datos históricos ---")
for tabla in tablas_a_validar:
    try:
        res = pd.read_sql(f"SELECT COUNT(*) as total FROM {tabla}", con=engine)
        print(f"Tabla: {tabla} | Total registros: {res['total'][0]:,}")
    except:
        print(f"Tabla: {tabla} | No se pudo acceder o no existe.")

--- Verificando volumen de datos históricos ---
Tabla: cm_fe_rips_cargas | Total registros: 2,033,166
Tabla: cm_rips_cargas | Total registros: 774,099
Tabla: aseg_afiliados | Total registros: 3,602,611
